# Merge Datasets — 2022 DHS Validation Variant

Builds model-ready numpy arrays for the Option 3 accuracy test
(2022 imagery vs 2022 DHS labels).

**Changes from phase7-merge-datasets (2025 version):**
- `DYNAMIC_CSV` points to `dynamic_features_2022_dhs_validation.csv`
- `MASTER_CSV` added — `master_cluster_summary.csv` used to extract
  the 317 DHS PointIDs (source == 'DHS')
- Section 3 filters `valid_ids` to DHS-origin points only before
  building arrays
- Output files named `*_2022_dhs.*` to avoid overwriting 2025 arrays
- Static features, feature names, and all preprocessing are identical

**Inputs:**
- `dynamic_features_2022_dhs_validation.csv` — from phase6 2022 variant
- `master_cluster_summary.csv` — source of DHS PointID list
- `static_osm_features_2025_prediction_points_expanded.csv` — unchanged
- `prediction_points.csv` — unchanged
- `static_feature_names.txt` — unchanged

**Outputs:**
- `X_dynamic_2022_dhs.npy` — shape (N, 4, 256)
- `X_static_2022_dhs.npy`  — shape (N, 52)
- `cluster_ids_2022_dhs.npy` — PointIDs in row order
- `static_feature_names.txt` — copied from training

In [2]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ============================================================
# 1. PATHS
# ============================================================

# CHANGED: dynamic features from 2022 DHS validation export
DYNAMIC_CSV = '/Users/ruben/Desktop/Thesis/2022Validation/dynamic_features_2022_dhs_validation.csv'

# NEW: master_cluster_summary.csv — used to get DHS PointID list
MASTER_CSV  = '/Users/ruben/Desktop/Thesis/2022Validation/master_cluster_summary.csv'

# UNCHANGED: same static features used for 2025 inference
STATIC_CSV  = '/Users/ruben/Desktop/Thesis/2022Validation/static_osm_features_2025_prediction_points_expanded.csv'
POINTS_CSV  = '/Users/ruben/Desktop/Thesis/2022Validation/prediction_points.csv'
NAMES_FILE  = '/Users/ruben/Desktop/Thesis/2022Validation/static_feature_names.txt'

# CHANGED: separate output folder to avoid overwriting 2025 arrays
OUTPUT_DIR  = '/Users/ruben/Desktop/Thesis/2022Validation/output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_COMPONENTS = 256

print('Paths configured.')

Paths configured.


## 2. Load Data

In [4]:
# ============================================================
# 2A. LOAD DYNAMIC FEATURES (2022)
# ============================================================

print('Loading dynamic features (2022, PCA-reduced)...')
df_dyn = pd.read_csv(DYNAMIC_CSV)
print(f'  Records       : {len(df_dyn)}')
print(f'  Columns       : {df_dyn.shape[1]}')
print(f'  Unique PointIDs: {df_dyn["PointID"].nunique()}')
print(f'  Quarters      : {sorted(df_dyn["Quarter"].unique())}')

cnn_cols = [c for c in df_dyn.columns if c.startswith('CNN_')]
print(f'  CNN feature columns: {len(cnn_cols)} (expected {N_COMPONENTS})')
assert len(cnn_cols) == N_COMPONENTS, (
    f'Expected {N_COMPONENTS} CNN columns, got {len(cnn_cols)}.'
)

Loading dynamic features (2022, PCA-reduced)...
  Records       : 1296
  Columns       : 258
  Unique PointIDs: 324
  Quarters      : ['Q1', 'Q2', 'Q3', 'Q4']
  CNN feature columns: 256 (expected 256)


In [5]:
# ============================================================
# 2B. LOAD DHS POINT IDs FROM MASTER CLUSTER SUMMARY
# ============================================================
# NEW: pull the 317 PointIDs where source == 'DHS'.
# These are the only points for which we have Actual_Wealth
# labels to compare predictions against.

print('Loading DHS PointIDs from master_cluster_summary.csv...')
df_master = pd.read_csv(MASTER_CSV)

dhs_ids = set(
    df_master.loc[df_master['source'] == 'DHS', 'PointID'].astype(int)
)

print(f'  Total rows in master : {len(df_master)}')
print(f'  DHS-origin PointIDs  : {len(dhs_ids)}')
print(f'  Non-DHS rows (grid)  : {(df_master["source"] != "DHS").sum()}')

Loading DHS PointIDs from master_cluster_summary.csv...
  Total rows in master : 1970
  DHS-origin PointIDs  : 317
  Non-DHS rows (grid)  : 1653


In [6]:
# ============================================================
# 2C. LOAD STATIC FEATURES (unchanged from 2025 version)
# ============================================================

print('Loading static features (OSM + VIIRS)...')
df_stat = pd.read_csv(STATIC_CSV)
print(f'  Records: {len(df_stat)}')

with open(NAMES_FILE) as f:
    training_feature_names = [line.strip() for line in f]
print(f'\nTraining feature order ({len(training_feature_names)} features):')
for i, name in enumerate(training_feature_names):
    present = 'OK' if name in df_stat.columns else 'MISSING'
    print(f'  {i+1:2d}. {name:30s} [{present}]')

Loading static features (OSM + VIIRS)...
  Records: 2000

Training feature order (52 features):
   1. Total_Road_Length              [OK]
   2. Main_Roads_Length              [OK]
   3. Secondary_Roads_Length         [OK]
   4. Local_Roads_Length             [OK]
   5. Tracks_Length                  [OK]
   6. Total_Bldg_Count               [OK]
   7. Total_Bldg_Area                [OK]
   8. Mean_Bldg_Area                 [OK]
   9. Total_Bldg_Proportion          [OK]
  10. Bldg_Density_per_km2           [OK]
  11. Bldg_residential_Count         [OK]
  12. Bldg_residential_TotalArea     [OK]
  13. Bldg_residential_MeanArea      [OK]
  14. Bldg_residential_Proportion    [OK]
  15. Bldg_commercial_Count          [OK]
  16. Bldg_commercial_TotalArea      [OK]
  17. Bldg_commercial_MeanArea       [OK]
  18. Bldg_commercial_Proportion     [OK]
  19. Bldg_industrial_Count          [OK]
  20. Bldg_industrial_TotalArea      [OK]
  21. Bldg_industrial_MeanArea       [OK]
  22. Bldg_industrial_

In [7]:
# ============================================================
# 2D. LOAD PREDICTION POINTS METADATA
# ============================================================

print('Loading prediction points...')
df_points = pd.read_csv(POINTS_CSV)
print(f'  Total points: {len(df_points)}')
print(f'  Provinces   : {df_points["Province"].nunique()}')

Loading prediction points...
  Total points: 2000
  Provinces   : 16


## 3. Identify Valid Points

In [8]:
# ============================================================
# 3. FIND DHS POINTS WITH COMPLETE DATA IN BOTH SETS
# ============================================================
# CHANGED: final valid_ids is the intersection of
#   (1) DHS-origin PointIDs from master_cluster_summary
#   (2) PointIDs with all 4 quarters in dynamic features
#   (3) PointIDs present in static features
# This guarantees only the 317 DHS points enter the arrays.

# (1) DHS IDs with all 4 quarters in dynamic
quarter_counts   = df_dyn.groupby('PointID').size()
complete_dynamic = set(quarter_counts[quarter_counts == 4].index)

# (2) Static ID column
if 'PointID' in df_stat.columns:
    stat_id_col = 'PointID'
elif 'DHSCLUST' in df_stat.columns:
    stat_id_col = 'DHSCLUST'
else:
    raise ValueError(
        f'Cannot find ID column in static CSV. '
        f'Columns: {df_stat.columns.tolist()}'
    )
available_static = set(df_stat[stat_id_col].unique())

# (3) Triple intersection: DHS ∩ complete dynamic ∩ available static
valid_ids = sorted(dhs_ids & complete_dynamic & available_static)

print(f'DHS PointIDs (from master)         : {len(dhs_ids)}')
print(f'Complete dynamic (4 quarters)      : {len(complete_dynamic)}')
print(f'Available in static                : {len(available_static)}')
print(f'Valid (DHS ∩ dynamic ∩ static)     : {len(valid_ids)}')

# Report any DHS points that dropped out
missing_dynamic = dhs_ids - complete_dynamic
missing_static  = dhs_ids - available_static
if missing_dynamic:
    print(f'\nDHS points missing from dynamic    : {len(missing_dynamic)}')
    print(f'  PointIDs: {sorted(missing_dynamic)}')
if missing_static:
    print(f'DHS points missing from static     : {len(missing_static)}')
    print(f'  PointIDs: {sorted(missing_static)}')

print(f'\nExpected: ~317. Actual: {len(valid_ids)}')

DHS PointIDs (from master)         : 317
Complete dynamic (4 quarters)      : 324
Available in static                : 2000
Valid (DHS ∩ dynamic ∩ static)     : 317

Expected: ~317. Actual: 317


## 4. Reshape Dynamic Features

In [9]:
# ============================================================
# 4. RESHAPE DYNAMIC INTO (N, 4, 256)
# ============================================================
# Already PCA-reduced from phase6 2022 variant. No PCA needed.
# Identical to phase7 logic.

print('Reshaping dynamic features...')

X_dynamic              = []
valid_clusters_ordered = []
skipped                = []

for pid in valid_ids:
    subset = df_dyn[
        df_dyn['PointID'] == pid
    ].sort_values('Quarter')

    if len(subset) != 4:
        skipped.append((pid, len(subset)))
        continue

    feats = subset[cnn_cols].values   # (4, 256)
    X_dynamic.append(feats)
    valid_clusters_ordered.append(pid)

X_dynamic = np.array(X_dynamic, dtype=np.float32)

print(f'X_dynamic shape : {X_dynamic.shape}')
print(f'  Expected      : ({len(valid_ids)}, 4, {N_COMPONENTS})')
print(f'  Skipped       : {len(skipped)}')
if skipped:
    print(f'  Skipped IDs   : {skipped}')

Reshaping dynamic features...
X_dynamic shape : (317, 4, 256)
  Expected      : (317, 4, 256)
  Skipped       : 0


## 5. Align Static Features

In [10]:
# ============================================================
# 5. ALIGN STATIC FEATURES IN TRAINING COLUMN ORDER
# ============================================================
# Identical to phase7. Log transforms, column ordering,
# and missing column handling are unchanged.

print('Aligning static features...')

log_cols = [
    'LU_Residential_m2', 'LU_Commercial_m2',
    'LU_Industrial_m2',  'LU_Agricultural_m2',
    'LU_Forest_m2',
    'POI_restaurant_Count', 'Total_POI_Count',
]
for col in log_cols:
    if col in df_stat.columns:
        df_stat[col] = np.log1p(df_stat[col])

df_stat_aligned = (
    df_stat[df_stat[stat_id_col].isin(valid_clusters_ordered)]
    .set_index(stat_id_col)
    .reindex(valid_clusters_ordered)
)

missing_cols = [
    c for c in training_feature_names
    if c not in df_stat_aligned.columns
]
extra_cols = [
    c for c in df_stat_aligned.columns
    if c not in training_feature_names
]

if missing_cols:
    print(f'  WARNING: Missing columns (zero-filled): {missing_cols}')
    for col in missing_cols:
        df_stat_aligned[col] = 0.0
if extra_cols:
    print(f'  Extra columns (dropped): {extra_cols}')

df_stat_aligned = df_stat_aligned[training_feature_names]
X_static = df_stat_aligned.values.astype(np.float32)

print(f'X_static shape  : {X_static.shape}')
print(f'  Expected      : ({len(valid_clusters_ordered)}, {len(training_feature_names)})')
print(f'  Column order  : {list(df_stat_aligned.columns)[:5]} ...')

Aligning static features...
X_static shape  : (317, 52)
  Expected      : (317, 52)
  Column order  : ['Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length'] ...


## 6. Validate

In [11]:
# ============================================================
# 6. VALIDATE MERGED DATASET
# ============================================================

cluster_ids = np.array(valid_clusters_ordered)

print('Validating merged dataset...')

assert X_dynamic.shape[0] == X_static.shape[0] == len(cluster_ids), (
    f'Row mismatch: dynamic={X_dynamic.shape[0]}, '
    f'static={X_static.shape[0]}, ids={len(cluster_ids)}'
)
print(f'  Row alignment : OK ({len(cluster_ids)} points)')

nan_dyn = np.isnan(X_dynamic).sum()
if nan_dyn > 0:
    print(f'  WARNING: {nan_dyn} NaN in dynamic. Filling with 0.')
    X_dynamic = np.nan_to_num(X_dynamic, nan=0.0)
else:
    print(f'  Dynamic NaN   : none')

nan_stat = np.isnan(X_static).sum()
if nan_stat > 0:
    print(f'  WARNING: {nan_stat} NaN in static. Filling with column means.')
    col_means = np.nanmean(X_static, axis=0)
    for j in range(X_static.shape[1]):
        mask = np.isnan(X_static[:, j])
        X_static[mask, j] = col_means[j]
else:
    print(f'  Static NaN    : none')

# Confirm all cluster_ids are DHS-origin
non_dhs = set(cluster_ids) - dhs_ids
if non_dhs:
    print(f'  WARNING: {len(non_dhs)} non-DHS PointIDs in output: {non_dhs}')
else:
    print(f'  DHS-only check: OK — all {len(cluster_ids)} PointIDs are DHS-origin')

print(f'\nFinal dataset:')
print(f'  X_dynamic    : {X_dynamic.shape} (PCA-reduced to {N_COMPONENTS} dims)')
print(f'  X_static     : {X_static.shape}')
print(f'  cluster_ids  : {cluster_ids.shape}')
print(f'  Dynamic range: [{X_dynamic.min():.2f}, {X_dynamic.max():.2f}]')
print(f'  Static range : [{X_static.min():.2f}, {X_static.max():.2f}]')

# Province breakdown
df_pts = df_points[df_points['PointID'].isin(cluster_ids)]
print(f'\nPoints per province (DHS only):')
for prov, count in df_pts['Province'].value_counts().items():
    print(f'  {prov}: {count}')

Validating merged dataset...
  Row alignment : OK (317 points)
  Dynamic NaN   : none
  Static NaN    : none
  DHS-only check: OK — all 317 PointIDs are DHS-origin

Final dataset:
  X_dynamic    : (317, 4, 256) (PCA-reduced to 256 dims)
  X_static     : (317, 52)
  cluster_ids  : (317,)
  Dynamic range: [-41.91, 104.68]
  Static range : [0.00, 7243856.50]

Points per province (DHS only):
  NCR: 126
  Pampanga: 20
  Zambales: 20
  Benguet: 18
  Basilan: 12
  Occidental Mindoro: 12
  Oriental Mindoro: 12
  Ilocos Norte: 12
  Aklan: 12
  Zamboanga del Norte: 12
  Davao Oriental: 12
  Bulacan: 12
  Tawi-Tawi: 11
  Kalinga: 10
  Bataan: 10
  Maguindanao del Sur: 6


## 7. Save

In [12]:
# ============================================================
# 7. SAVE NUMPY ARRAYS
# ============================================================
# CHANGED: filenames include '_2022_dhs' suffix to distinguish
# from the 2025 inference arrays.

np.save(f'{OUTPUT_DIR}/X_dynamic_2022_dhs.npy',  X_dynamic)
np.save(f'{OUTPUT_DIR}/X_static_2022_dhs.npy',   X_static)
np.save(f'{OUTPUT_DIR}/cluster_ids_2022_dhs.npy', cluster_ids)

import shutil
shutil.copy(NAMES_FILE, f'{OUTPUT_DIR}/static_feature_names.txt')

print(f'Saved to: {OUTPUT_DIR}')
print(f'  X_dynamic_2022_dhs.npy    ({X_dynamic.nbytes / 1e6:.1f} MB)')
print(f'  X_static_2022_dhs.npy     ({X_static.nbytes / 1e6:.1f} MB)')
print(f'  cluster_ids_2022_dhs.npy  ({cluster_ids.nbytes / 1e3:.1f} KB)')
print(f'  static_feature_names.txt')

Saved to: /Users/ruben/Desktop/Thesis/2022Validation/output/
  X_dynamic_2022_dhs.npy    (1.3 MB)
  X_static_2022_dhs.npy     (0.1 MB)
  cluster_ids_2022_dhs.npy  (2.5 KB)
  static_feature_names.txt


In [13]:
# ============================================================
# 7B. SAVE y_wealth ARRAY
# ============================================================
# Actual_Wealth from master_cluster_summary.csv, aligned to
# the same row order as cluster_ids_2022_dhs.npy.

df_wealth = (
    df_master[df_master['source'] == 'DHS']
    .set_index('PointID')['Actual_Wealth']
)

# Reindex to match the exact row order of cluster_ids
y_wealth = df_wealth.reindex(valid_clusters_ordered).values.astype(np.float32)

# Sanity checks
assert len(y_wealth) == len(cluster_ids), (
    f'Length mismatch: y_wealth={len(y_wealth)}, cluster_ids={len(cluster_ids)}'
)
missing_labels = np.isnan(y_wealth).sum()
if missing_labels > 0:
    print(f'WARNING: {missing_labels} NaN in y_wealth — '
          f'these PointIDs have no Actual_Wealth in master_cluster_summary.')
    print('Affected PointIDs:')
    for pid, val in zip(valid_clusters_ordered, y_wealth):
        if np.isnan(val):
            print(f'  PointID {pid}')
else:
    print(f'y_wealth: all {len(y_wealth)} values present, no NaN')

np.save(f'{OUTPUT_DIR}/y_wealth_2022_dhs.npy', y_wealth)

print(f'\ny_wealth_2022_dhs.npy saved')
print(f'  Shape : {y_wealth.shape}')
print(f'  Range : [{np.nanmin(y_wealth):.4f}, {np.nanmax(y_wealth):.4f}]')
print(f'  Mean  : {np.nanmean(y_wealth):.4f}')
print(f'  Std   : {np.nanstd(y_wealth):.4f}')

y_wealth: all 317 values present, no NaN

y_wealth_2022_dhs.npy saved
  Shape : (317,)
  Range : [-2.1489, 1.8740]
  Mean  : 0.3325
  Std   : 0.7536


In [ ]:
# ============================================================
# 8. UPLOAD TO GCS (optional)
# ============================================================

import subprocess
GCS_DEST = 'gs://tala-sentinel2-data/data/merged_2022_dhs'
for fname in [
    'X_dynamic_2022_dhs.npy',
    'X_static_2022_dhs.npy',
    'cluster_ids_2022_dhs.npy',
    'static_feature_names.txt'
]:
    subprocess.run([
        'gsutil', 'cp',
        f'{OUTPUT_DIR}/{fname}',
        f'{GCS_DEST}/{fname}'
    ], check=True)
    print(f'  Uploaded: {fname}')
print(f'All files at: {GCS_DEST}')

## 9. Note for Inference Script

When running the LSTM inference on these arrays, point the script at
the `_2022_dhs` files. The same note from phase7 applies:
dynamic features are already PCA-reduced so skip the PCA transform:

```python
X_dynamic = np.load('X_dynamic_2022_dhs.npy')   # (N, 4, 256)
X_static  = np.load('X_static_2022_dhs.npy')    # (N, 52)
ids       = np.load('cluster_ids_2022_dhs.npy') # (N,)

# Dynamic features are already PCA-reduced — do NOT apply pca.transform()
N_COMPONENTS  = X_dynamic.shape[2]   # 256
X_dynamic_pca = X_dynamic            # pass directly to model
```

After inference, compare predicted WI against `Actual_Wealth` in
`master_cluster_summary.csv` using `cluster_ids_2022_dhs.npy` as
the join key. That gives you the 2022 MAE, RMSE, and Pearson r
to place alongside the 2025 figures (MAE=0.775, r=0.477, RMSE=0.946).